# Lesson 1.5 — Vision Input and Prompt Caching

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Send image (vision) input via base64 content blocks
- Apply prompt caching with cache_control to reuse large stable prefixes
- Read cache creation vs cache read token usage
- Identify when caching lowers cost and latency

**You will produce:** A vision request builder and a cached-system-prompt demo that reports cache usage.

## Concepts

The Messages API accepts **multi-format input**. For **vision**, add an `image` content block with base64 data (or a URL source where supported) alongside text in the same user message.

**Prompt caching** lets you mark a large, stable prefix (system instructions, documents, tool definitions) with `cache_control` so Claude reuses it across requests at reduced cost and latency. The first call pays a cache-write; later calls within the cache lifetime pay a much cheaper cache-read. Usage fields report `cache_creation_input_tokens` and `cache_read_input_tokens`.

> Cache minimum sizes, lifetimes, and the `cache_control` placement rules change over time - verify current limits at https://docs.anthropic.com/en/docs/build-with-claude/prompt-caching.

> **Reference:** [Anthropic - Prompt caching](https://docs.anthropic.com/en/docs/build-with-claude/prompt-caching)

In [ ]:
# !pip install anthropic
import os

try:
    import anthropic
except ImportError:
    anthropic = None
    print("anthropic SDK not installed. Run: pip install anthropic")

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
HAVE_API = bool(API_KEY) and anthropic is not None

client = anthropic.Anthropic() if HAVE_API else None  # reads ANTHROPIC_API_KEY

# Current model aliases (confirm latest at https://docs.anthropic.com/en/docs/about-claude/models)
MODEL_HAIKU = "claude-haiku-4-5"
MODEL_SONNET = "claude-sonnet-4-5"
MODEL_OPUS = "claude-opus-4-1"

def mock_text(prompt):
    """Offline fallback so the notebook runs without an API key."""
    return f"[MOCK RESPONSE] Set ANTHROPIC_API_KEY for a live call. Prompt was: {prompt[:80]}..."

print("API ready" if HAVE_API else "No API key - cells will print mock responses")

## Code-Along: Step 1 — Build a vision message

Combine an image block and a text block in one user turn.

In [ ]:
import base64

def vision_message(image_bytes, question, media_type="image/png"):
    b64 = base64.standard_b64encode(image_bytes).decode("utf-8")
    return [{
        "role": "user",
        "content": [
            {"type": "image", "source": {"type": "base64",
                "media_type": media_type, "data": b64}},
            {"type": "text", "text": question},
        ],
    }]

payload = vision_message(b"\x89PNG...fake...", "What is in this image?")
print("Built vision payload with", len(payload[0]['content']), "blocks")

## Code-Along: Step 2 — Cache a large system prefix

Mark a big, reusable instruction block with cache_control.

In [ ]:
BIG_SYSTEM = "You are a policy assistant. " + ("Policy clause. " * 400)

def cached_call(user):
    if not HAVE_API:
        print("[MOCK] cache_creation=... cache_read=..."); return mock_text(user)
    msg = client.messages.create(
        model=MODEL_SONNET, max_tokens=100,
        system=[{"type": "text", "text": BIG_SYSTEM,
                 "cache_control": {"type": "ephemeral"}}],
        messages=[{"role": "user", "content": user}],
    )
    print("usage:", msg.usage)
    return msg.content[0].text

cached_call("Summarize clause 1.")  # first call: cache write
cached_call("Summarize clause 2.")  # second call: cache read (cheaper)

## Exercise

Write `cache_savings(creation, read, write_mult=1.25, read_mult=0.1)` that estimates the fraction of input cost saved by caching vs re-sending the prefix each time. Return a percentage.

In [ ]:
def cache_savings(creation_tokens, read_tokens, write_mult=1.25, read_mult=0.1):
    # TODO: baseline cost = (creation_tokens + read_tokens) at full price
    # TODO: cached cost = creation_tokens*write_mult + read_tokens*read_mult
    # TODO: return percent saved
    pass

## Key Takeaways

- Vision input is a base64 image content block sent alongside text in a user message.
- Prompt caching reuses a stable prefix via cache_control to cut cost and latency.
- First cached call pays a write; subsequent reads are much cheaper.
- Check cache_creation_input_tokens and cache_read_input_tokens to confirm cache hits.

---

**Next:** [Lesson 1.6 — Model Selection, Batch vs Realtime, and Token Cost](./lesson-1.6-model-selection-cost.ipynb)